# Maritime VHF ASR — Dissertation Notebook
Transformer-based ASR fine-tuning on real and simulated maritime VHF speech.

Before starting: Runtime → Change runtime type → T4 GPU

| Model | GPU |
|---|---|
| Whisper-small / Wav2Vec2 | T4 (free tier) |
| Whisper-medium | T4 (~30 min / experiment) |
| Whisper-large | A100 (Colab Pro+) |

## Step 1 — Environment

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"{gpu.name}  {gpu.total_memory/1e9:.1f} GB")
else:
    print("No GPU — Runtime → Change runtime type → T4")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Clone / Update Repository

Run at the start of every session and after pushing local changes.

Checkpoints, manifests, and results persist on Drive at `/content/drive/MyDrive/ASR_Dissertation/`.
The `/content/asr_dissertation` directory is recreated each session by this cell.

In [ ]:
import os

REPO    = "https://github.com/tolgaSarmi/maritime_asr.git"
WORKDIR = "/content/asr_dissertation"

if os.path.exists(WORKDIR):
    !git -C {WORKDIR} pull
else:
    !git clone {REPO} {WORKDIR}

os.chdir(WORKDIR)
print(os.getcwd())

## Step 3 — Install Dependencies

Run Cell A, then **Runtime → Restart session**, then Cell B and Cell C. Skip Cell A in future sessions unless packages have changed.

In [ ]:
# Cell A
!pip install -q \
    "scipy>=1.14.0" "scikit-learn>=1.6.0" "torchao>=0.16.0" \
    transformers peft datasets accelerate evaluate \
    jiwer librosa soundfile omegaconf rich inflect \
    tensorboard seaborn
print("Done — restart the session now")

In [ ]:
# Cell B
import os
os.chdir('/content/asr_dissertation')
print(os.getcwd())

In [ ]:
# Cell C
import numpy, scipy, transformers, peft, omegaconf, evaluate, torchao
from packaging.version import Version

for name, mod in [("numpy", numpy), ("scipy", scipy), ("transformers", transformers),
                  ("peft", peft), ("torchao", torchao)]:
    print(f"{name:15s} {mod.__version__}")

if Version(torchao.__version__) < Version("0.16.0"):
    raise RuntimeError(f"torchao {torchao.__version__} too old — re-run Cell A and restart")

print("\nEnvironment OK")

## Step 4 — Prepare Datasets

Validates records, normalises transcriptions, and creates 80/10/10 train/val/test splits.
Manifests are read from and written to Drive at `/content/drive/MyDrive/ASR_Dissertation/data/`.

In [ ]:
!python main.py --mode data

## Step 5 — Training

Training resumes automatically from the latest checkpoint if the session disconnects — just re-run the same cell.

Start with `ef_whisper_small_real` (~30 min on T4) to verify the setup, then run all experiments below.

In [ ]:
!python main.py --mode list

In [ ]:
EXPERIMENT = "ef_whisper_small_real"
!python main.py --mode train --experiment {EXPERIMENT}

In [ ]:
import subprocess, sys

EXPERIMENTS = [
    "ef_whisper_small_real",
    "ef_whisper_medium_real",
    "ef_wav2vec2_real",
    "lora_whisper_small_real",
    "lora_whisper_medium_real",
]

for exp in EXPERIMENTS:
    print(f"\n{'─'*50}\n{exp}\n{'─'*50}")
    r = subprocess.run([sys.executable, "main.py", "--mode", "train", "--experiment", exp])
    if r.returncode != 0:
        print(f"{exp} failed — continuing")

In [ ]:
import os

checkpoint_dir = '/content/drive/MyDrive/ASR_Dissertation/checkpoints'
for exp in sorted(os.listdir(checkpoint_dir)):
    path = os.path.join(checkpoint_dir, exp)
    if os.path.isdir(path):
        ckpts = sorted(d for d in os.listdir(path) if d.startswith('checkpoint-'))
        print(f"{exp}: {', '.join(ckpts) or 'none'}")

## Step 6 — Evaluate

In [ ]:
!python main.py --mode eval_all

In [ ]:
DRIVE      = '/content/drive/MyDrive/ASR_Dissertation'
CHECKPOINT = f"{DRIVE}/checkpoints/ef_whisper_small_real"
MANIFEST   = f"{DRIVE}/data/real/test_manifest.json"
!python main.py --mode test_wer --checkpoint {CHECKPOINT} --manifest {MANIFEST}

## Step 7 — Figures

In [ ]:
!python main.py --mode figures

In [ ]:
from IPython.display import Image, display
import glob

for fig in sorted(glob.glob('results/figures/*.png')):
    print(fig.split('/')[-1])
    display(Image(fig, width=900))

In [ ]:
import shutil
shutil.copytree('results', '/content/drive/MyDrive/ASR_Dissertation/results', dirs_exist_ok=True)

## Step 8 — Results Table

In [ ]:
import json, pandas as pd

with open('results/all_results.json') as f:
    all_results = json.load(f)

rows = []
for exp_name, res in all_results.items():
    for key, val in res.items():
        if key.startswith('eval_') and isinstance(val, dict) and 'wer' in val:
            rows.append({
                'experiment': exp_name,
                'method'    : res.get('method', ''),
                'train_data': res.get('train_data', '—'),
                'eval_set'  : key[5:],
                'wer'       : round(val['wer'] * 100, 2),
                'cer'       : round(val.get('cer', 0) * 100, 2),
            })

df = pd.DataFrame(rows).sort_values(['method', 'eval_set', 'wer'])
pd.set_option('display.max_rows', 100)
print(df.to_string(index=False))

## Bonus — Transcribe a Single File

In [ ]:
from src.inference import build_transcriber
from google.colab import files

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

CHECKPOINT = "/content/drive/MyDrive/ASR_Dissertation/checkpoints/ef_whisper_small_real"
transcriber = build_transcriber("whisper", CHECKPOINT)
text, elapsed = transcriber.transcribe_file(audio_path)

print(f"Transcription : {text}")
print(f"Latency       : {elapsed*1000:.0f} ms")

---
## Session Reset

1. Step 1 — GPU check and Drive mount
2. Step 2 — pull latest code
3. Step 3 — Cell B, Cell C (skip Cell A unless packages changed)
4. Step 4 — run data pipeline
5. Step 5 — continue training (resumes from last checkpoint automatically)

---
## Out of Memory

Switch to A100, or lower batch size in `configs/config.yaml`:

```yaml
per_device_train_batch_size: 2
gradient_accumulation_steps: 16
```